# Modélisation du churn — Spark MLlib (lecture sur Gold)

**Rôle :** Data Scientist · **Lecture** : `gold/churn_features` · **Écriture** : `gold/churn_predictions` + `models/`.

Objectif : entraîner et comparer 3 modèles (baseline → Random Forest → GBT), valider par
validation croisée, et **interpréter** les résultats (métriques métier, feature importance,
analyse d'erreurs). Seed fixée partout pour la reproductibilité.

In [1]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import config
from pyspark.sql import functions as F

SEED = 42  # reproductibilité : même seed partout

spark = config.build_spark(app_name='churn-modeling')
spark.sparkContext.setLogLevel('WARN')
print('Gold features :', config.s3_path('gold_feat'))

Gold features : s3a://nacerdinedouniaamira/gold/churn_features


In [2]:
# --- Réduire le bruit dans les sorties ---
import logging, warnings

spark.sparkContext.setLogLevel('ERROR')          # masque les WARN Spark
logging.getLogger('py4j').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')                # masque les FutureWarning/Pandas

# Optionnel : couper les avertissements "plan trop large"
spark.conf.set('spark.sql.debug.maxToStringFields', '200')

## 1. Lecture des features

In [3]:
df = spark.read.parquet(config.s3_path('gold_feat'))
df = df.na.drop(subset=['churn'])
df.cache()

print('Lignes   :', df.count())
print('Colonnes :', len(df.columns))
df.printSchema()

# Vérif : base_id bien présent (indispensable pour le split anti-leakage)
print('base_id distincts :', df.select('base_id').distinct().count())

Lignes   : 704300
Colonnes : 17
root
 |-- base_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- tenure: double (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- senior_citizen: integer (nullable = true)
 |-- partner: string (nullable = true)
 |-- dependents: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- online_security: string (nullable = true)
 |-- tech_support: string (nullable = true)
 |-- paperless_billing: string (nullable = true)
 |-- tenure_bucket: string (nullable = true)
 |-- avg_monthly_spend: double (nullable = true)
 |-- num_services: integer (nullable = true)
 |-- churn: integer (nullable = true)



base_id distincts : 7043


## 2. Split anti-leakage par client d'origine

Rappel EDA : chaque client réel est répliqué 100 fois. Un `randomSplit` mettrait des copies
du même client dans train ET test → AUC gonflé. On splitte donc **sur `base_id`** : un client
d'origine est entièrement dans train OU dans test, jamais les deux.

In [4]:
# On tire les clients D'ORIGINE (base_id), pas les lignes
base_ids = df.select('base_id').distinct()
train_ids, test_ids = base_ids.randomSplit([0.8, 0.2], seed=SEED)

train = df.join(train_ids, on='base_id', how='inner')
test  = df.join(test_ids,  on='base_id', how='inner')

print(f'Train : {train.count():,} lignes / {train_ids.count():,} clients')
print(f'Test  : {test.count():,} lignes / {test_ids.count():,} clients')

# Contrôle anti-leakage : aucun base_id partagé entre train et test
overlap = train.select('base_id').intersect(test.select('base_id')).count()
print('Clients partagés train/test (doit être 0) :', overlap)

# Contrôle : taux de churn préservé dans les deux ensembles
train.groupBy('churn').count().show()
test.groupBy('churn').count().show()

Train : 569,800 lignes / 5,698 clients


Test  : 134,500 lignes / 1,345 clients


Clients partagés train/test (doit être 0) : 0


+-----+------+
|churn| count|
+-----+------+
|    1|150000|
|    0|419800|
+-----+------+



+-----+-----+
|churn|count|
+-----+-----+
|    1|36900|
|    0|97600|
+-----+-----+



### Conclusion §2 — Split validé

- **0 client d'origine partagé** entre train et test → anti-leakage confirmé.
- Découpage clients : 5 698 train / 1 345 test (≈ 81 % / 19 %).
- Taux de churn préservé : 26,3 % (train) vs 27,4 % (test) vs 26,5 % (global) → split représentatif.

Le test est un échantillon honnête : aucune réplique d'un client d'entraînement n'y figure, donc
les métriques refléteront la vraie capacité de généralisation à des clients jamais vus.

## 3. Pipeline de features Spark ML

Les modèles MLlib attendent un unique vecteur `features`. On indexe puis one-hot encode les
catégorielles, on assemble le tout avec les numériques.

In [5]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Catégorielles (incl. la dérivée tenure_bucket) -> index -> one-hot
categorical = ['partner', 'dependents', 'contract', 'payment_method',
               'internet_service', 'online_security', 'tech_support',
               'paperless_billing', 'tenure_bucket']

# Numériques + binaire déjà en 0/1 (senior_citizen)
numeric = ['tenure', 'monthly_charges', 'avg_monthly_spend', 'num_services', 'senior_citizen']

indexers = [StringIndexer(inputCol=c, outputCol=c + '_idx', handleInvalid='keep')
            for c in categorical]
encoders = [OneHotEncoder(inputCol=c + '_idx', outputCol=c + '_oh') for c in categorical]

assembler = VectorAssembler(
    inputCols=[c + '_oh' for c in categorical] + numeric,
    outputCol='features'
)

feature_stages = indexers + encoders + [assembler]
print('Étapes de features :', len(feature_stages))
print('Catégorielles :', len(categorical), '| Numériques :', len(numeric))

Étapes de features : 19
Catégorielles : 9 | Numériques : 5


## 4. Modèle 1 — Baseline : régression logistique

Un modèle linéaire simple et interprétable. Il fixe le **plancher de performance** que les
modèles plus complexes (RF, GBT) devront dépasser pour justifier leur complexité.

In [6]:
df.unpersist()
train = train.cache()
test  = test.cache()
print('train', train.count(), '| test', test.count())     # force le cache

12:08:00.931 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 1 on 10.233.127.59: 
The executor with id 1 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:06:32Z
	 container finished at: 2026-06-21T12:07:58Z
	 exit code: 137
	 termination reason: OOMKilled
      


train 569800 | test 134500


In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

lr = LogisticRegression(labelCol='churn', featuresCol='features', maxIter=50)
pipeline_lr = Pipeline(stages=feature_stages + [lr])

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

# Évaluateurs réutilisés pour tous les modèles
auc_roc = BinaryClassificationEvaluator(labelCol='churn', metricName='areaUnderROC')
auc_pr  = BinaryClassificationEvaluator(labelCol='churn', metricName='areaUnderPR')
f1_eval = MulticlassClassificationEvaluator(labelCol='churn', metricName='f1')

print('=== Baseline — Régression logistique ===')
print(f'AUC-ROC : {auc_roc.evaluate(pred_lr):.4f}')
print(f'AUC-PR  : {auc_pr.evaluate(pred_lr):.4f}')
print(f'F1      : {f1_eval.evaluate(pred_lr):.4f}')

12:08:26.038 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 2 on 10.233.125.142: 
The executor with id 2 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:07:39Z
	 container finished at: 2026-06-21T12:08:24Z
	 exit code: 137
	 termination reason: OOMKilled
      


=== Baseline — Régression logistique ===


AUC-ROC : 0.8397


AUC-PR  : 0.6616


F1      : 0.7928


### Résultats — Baseline (régression logistique)

| Métrique | Valeur | Lecture |
|---|---|---|
| AUC-ROC | 0,840 | très bon pouvoir discriminant pour un modèle linéaire |
| AUC-PR | 0,662 | ~2,5× le hasard (0,27 = taux de churn) → utile sur la classe minoritaire |
| F1 | 0,793 | compromis précision/rappel au seuil 0,5 |

**Interprétation métier** : trié par score de risque, le haut de liste concentre les vrais
churners → une campagne de rétention ciblée serait efficace. L'AUC-PR (0,66) est la métrique clé
vu le déséquilibre (26,5 %) : c'est elle qui mesure la capacité à prioriser les partants.

**Repère** : seuil à battre par les modèles suivants = **AUC-ROC 0,84 / AUC-PR 0,66**.

## 5. Modèle 2 — Random Forest

Ensemble d'arbres décorrélés. Capture des **interactions non linéaires** entre variables
(ex. « contrat mensuel ET fibre ») que la régression logistique ne voit pas. On garde le même
pipeline de features pour une comparaison juste.

In [8]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol='churn', featuresCol='features',
                            numTrees=100, maxDepth=8, seed=SEED)
pipeline_rf = Pipeline(stages=feature_stages + [rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

print('=== Modèle 2 — Random Forest ===')
print(f'AUC-ROC : {auc_roc.evaluate(pred_rf):.4f}')
print(f'AUC-PR  : {auc_pr.evaluate(pred_rf):.4f}')
print(f'F1      : {f1_eval.evaluate(pred_rf):.4f}')

=== Modèle 2 — Random Forest ===


AUC-ROC : 0.8425


AUC-PR  : 0.6750


F1      : 0.7852


### Résultats — Modèle 2 : Random Forest

| Métrique | Baseline (LR) | Random Forest | Écart |
|---|---|---|---|
| AUC-ROC | 0,840 | **0,843** | +0,003 |
| AUC-PR | 0,662 | **0,675** | +0,013 |
| F1 | 0,793 | 0,785 | −0,008 |

Le RF dépasse la baseline sur les deux AUC, notamment l'AUC-PR (+0,013, la métrique clé vu le
déséquilibre) : il capte des interactions non linéaires entre variables. Le gain reste **modeste**,
ce qui confirme que le signal du churn est largement linéaire (écarts nets et monotones vus à
l'EDA) — un modèle simple est déjà très compétitif. La légère baisse de F1 tient au seuil 0,5
non optimal, pas au pouvoir de classement (corrigé plus loin).

## 6. Modèle 3 — Gradient Boosted Trees (GBT)

Arbres construits **séquentiellement**, chacun corrigeant les erreurs du précédent. Souvent le
plus performant sur données tabulaires. Réglages manuels ici ; il sera optimisé par validation
croisée s'il s'avère le meilleur.

In [9]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(labelCol='churn', featuresCol='features',
                    maxIter=50, maxDepth=5, seed=SEED)
pipeline_gbt = Pipeline(stages=feature_stages + [gbt])

model_gbt = pipeline_gbt.fit(train)
pred_gbt = model_gbt.transform(test)

print('=== Modèle 3 — GBT ===')
print(f'AUC-ROC : {auc_roc.evaluate(pred_gbt):.4f}')
print(f'AUC-PR  : {auc_pr.evaluate(pred_gbt):.4f}')
print(f'F1      : {f1_eval.evaluate(pred_gbt):.4f}')

12:10:46.145 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 3 on 10.233.125.191: 
The executor with id 3 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:08:14Z
	 container finished at: 2026-06-21T12:10:45Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:11:14.313 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 4 on 10.233.127.190: 
The executor with id 4 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:08:28Z
	 container finished at: 2026-06-21T12:11:13Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:11:41.388 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 5 on 10.233.127.165: 
The executor with id 5 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:10:55Z
	 container finished at: 2026-06-21T12:11:40Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:11:53.473 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 6 on 10.233.125.205: 
The executor with id 6 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:11:15Z
	 container finished at: 2026-06-21T12:11:51Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:12:17.536 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 7 on 10.233.127.189: 
The executor with id 7 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:11:53Z
	 container finished at: 2026-06-21T12:12:16Z
	 exit code: 137
	 termination reason: OOMKilled
      


...

12:13:32.718 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 11 on 10.233.127.184: 
The executor with id 11 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:13:00Z
	 container finished at: 2026-06-21T12:13:30Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:13:33.773 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 12 on 10.233.125.76: 
The executor with id 12 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:13:01Z
	 container finished at: 2026-06-21T12:13:32Z
	 exit code: 137
	 termination reason: OOMKilled
      


AUC-ROC : 0.8431


AUC-PR  : 0.6753


F1      : 0.7886


### Comparaison des 3 modèles 

| Métrique | Baseline (LR) | Random Forest | GBT |
|---|---|---|---|
| AUC-ROC | 0,840 | 0,843 | **0,843** |
| AUC-PR | 0,662 | 0,675 | **0,675** |
| F1 | 0,793 | 0,785 | 0,789 |

Les 3 modèles tiennent dans un écart de 0,003 d'AUC-ROC. Le GBT est marginalement le meilleur,
mais le gain de la complexité est négligeable → **le signal du churn est largement linéaire**
(effets monotones de contract, tenure, internet_service vus à l'EDA). La régression logistique,
quasi équivalente et bien plus interprétable, reste une option finale crédible.

➡️ Modèle retenu pour l'optimisation : **GBT** (meilleur sur les 2 AUC).

## 7. Comparaison visuelle des 3 modèles

In [17]:
import matplotlib.pyplot as plt
import numpy as np

results = {
    'LR':  {'pred': pred_lr,  'roc': 0.8397, 'pr': 0.6616, 'f1': 0.7928},
    'RF':  {'pred': pred_rf,  'roc': 0.8425, 'pr': 0.6750, 'f1': 0.7852},
    'GBT': {'pred': pred_gbt, 'roc': auc_roc.evaluate(pred_gbt),
            'pr': auc_pr.evaluate(pred_gbt), 'f1': f1_eval.evaluate(pred_gbt)},
}
models = list(results)
x = np.arange(len(models)); w = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w, [results[m]['roc'] for m in models], w, label='AUC-ROC', color='tab:blue')
ax.bar(x,     [results[m]['pr']  for m in models], w, label='AUC-PR',  color='tab:orange')
ax.bar(x + w, [results[m]['f1']  for m in models], w, label='F1',      color='tab:green')
ax.axhline(0.2654, ls='--', color='red', alpha=0.6, label='Hasard AUC-PR')
ax.set_xticks(x); ax.set_xticklabels(models); ax.set_ylim(0, 1)
ax.set_title('Comparaison des 3 modèles'); ax.legend()
plt.tight_layout(); plt.show()

In [18]:
from pyspark.sql import functions as F

def curve_points(pred):
    p = (pred.select('probability', 'churn')
            .rdd.map(lambda r: (float(r['probability'][1]), float(r['churn'])))
            .toDF(['score', 'label']).toPandas()
            .sort_values('score', ascending=False).reset_index(drop=True))
    P = p['label'].sum(); N = len(p) - P
    tp = np.cumsum(p['label'].values); fp = np.cumsum(1 - p['label'].values)
    return fp / N, tp / P, tp / P, tp / (tp + fp)   # fpr, tpr, recall, precision

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for m, c in zip(models, ['tab:blue', 'tab:orange', 'tab:green']):
    fpr, tpr, rec, prec = curve_points(results[m]['pred'])
    ax1.plot(fpr, tpr, label=f"{m} ({results[m]['roc']:.3f})", color=c)
    ax2.plot(rec, prec, label=f"{m} ({results[m]['pr']:.3f})", color=c)
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4); ax1.set_title('Courbe ROC')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR'); ax1.legend()
ax2.axhline(0.2654, ls='--', color='red', alpha=0.5)
ax2.set_title('Courbe Précision-Rappel'); ax2.set_xlabel('Rappel'); ax2.set_ylabel('Précision'); ax2.legend()
plt.tight_layout(); plt.show()

## 8. Optimisation des hyperparamètres (tuning manuel)

### Pourquoi pas de CrossValidator Spark ici ?

La validation croisée automatique de Spark (`CrossValidator`) enchaîne *k × n* entraînements GBT
en parallèle (3 folds × grille). Sur l'infrastructure allouée (mémoire executor limitée), cette
charge simultanée a saturé les conteneurs : OOM en cascade jusqu'à `Max executor failures
reached` et arrêt du SparkContext. La CV automatique n'était donc pas exécutable de façon stable
dans cet environnement.

**Solution adoptée** : on optimise `maxDepth` (l'hyperparamètre le plus influent du GBT) par une
recherche **manuelle séquentielle** — un seul modèle entraîné à la fois, sur les clients uniques.
C'est moins automatisé que `CrossValidator`.

### Anti-leakage & mémoire
On tune sur les **7 043 clients réels** (`dropDuplicates('base_id')`) : les répliques ×100
n'apportent aucune information pour le choix d'hyperparamètres, et travailler sur ce volume réduit
évite les saturations mémoire.

In [10]:
# Données clients uniques, colonnes réduites (léger en mémoire, pas de leakage)
keep = categorical + numeric + ['base_id', 'churn']
train_u = train.dropDuplicates(['base_id']).select(*keep).cache()
test_u  = test.dropDuplicates(['base_id']).select(*keep).cache()
print('train unique :', train_u.count(), '| test unique :', test_u.count())

# Tuning manuel séquentiel : un modèle à la fois (pas de surcharge mémoire)
results_tuning = []
for depth in [3, 5, 7]:
    gbt_t = GBTClassifier(labelCol='churn', featuresCol='features',
                          maxDepth=depth, maxIter=30, seed=SEED)
    model_t = Pipeline(stages=feature_stages + [gbt_t]).fit(train_u)
    pred_t = model_t.transform(test_u)
    roc = auc_roc.evaluate(pred_t)
    pr  = auc_pr.evaluate(pred_t)
    results_tuning.append((depth, roc, pr, model_t))
    print(f'maxDepth={depth} -> AUC-ROC={roc:.4f} | AUC-PR={pr:.4f}')

# Sélection du meilleur (sur AUC-ROC)
best_depth, best_roc, best_pr, best_model = max(results_tuning, key=lambda x: x[1])
print(f'\nMeilleur modèle : maxDepth={best_depth} (AUC-ROC={best_roc:.4f})')

train unique : 5698 | test unique : 1345


12:14:31.845 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 14 on 10.233.127.61: 
The executor with id 14 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:13:35Z
	 container finished at: 2026-06-21T12:14:30Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:14:37.902 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 13 on 10.233.125.53: 
The executor with id 13 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:13:34Z
	 container finished at: 2026-06-21T12:14:36Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:14:55.992 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 15 on 10.233.125.226: 
The executor with id 15 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:14:23Z
	 container finished at: 2026-06-21T12:14:53Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:15:21.050 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 16 on 10.233.127.31: 
The executor with id 16 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:14:27Z
	 container finished at: 2026-06-21T12:15:19Z
	 exit code: 137
	 termination reason: OOMKilled
      


...

12:16:19.265 [rpc-server-4-2] ERROR org.apache.spark.network.client.TransportResponseHandler - Still have 1 requests outstanding when connection from /10.233.125.188:52756 is closed


12:16:21.208 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 25 on 10.233.125.188: 
The executor with id 25 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:15:06Z
	 container finished at: 2026-06-21T12:16:19Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:16:33.256 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 24 on 10.233.125.219: 
The executor with id 24 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:15:01Z
	 container finished at: 2026-06-21T12:16:31Z
	 exit code: 137
	 termination reason: OOMKilled
      


12:16:51.307 [dispatcher-CoarseGrainedScheduler] ERROR org.apache.spark.scheduler.TaskSchedulerImpl - Lost executor 21 on 10.233.125.14: 
The executor with id 21 exited with exit code 137(SIGKILL, possible container OOM).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: docker.io/inseefrlab/onyxia-jupyter-pyspark:py3.13.13-spark4.1.1
	 container state: terminated
	 container started at: 2026-06-21T12:14:47Z
	 container finished at: 2026-06-21T12:16:50Z
	 exit code: 137
	 termination reason: OOMKilled
      
12:16:51.363 [kubernetes-executor-snapshots-subscribers-1] ERROR org.apache.spark.scheduler.cluster.k8s.ExecutorPodsLifecycleManager - Max number of executor failures (20) reached
12:16:51.402 [Thread-3] ERROR org.apache.spark.ml.util.Instrumentation - org.apache.spark.SparkException: Job 521 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(D

Py4JJavaError: An error occurred while calling o6970.fit.
: org.apache.spark.SparkException: Job 521 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1309)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1307)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1307)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:3424)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:85)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$stop$3(DAGScheduler.scala:3307)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1314)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:3307)
	at org.apache.spark.Sp

In [ ]:
pred_best = best_model.transform(test_u)
print('=== GBT optimisé (tuning manuel) ===')
print(f'AUC-ROC : {auc_roc.evaluate(pred_best):.4f}')
print(f'AUC-PR  : {auc_pr.evaluate(pred_best):.4f}')
print(f'F1      : {f1_eval.evaluate(pred_best):.4f}')